<a href="https://colab.research.google.com/github/switlanakostyuk-ctrl/Apollo/blob/main/%22HW_15_3_%D0%A2%D0%B5%D1%81%D1%82%D0%B8_%D0%B4%D0%BB%D1%8F_%D0%BC%D0%B0%D0%BB%D0%B8%D1%85_%D0%B2%D0%B8%D0%B1%D1%96%D1%80%D0%BE%D0%BA_%D1%82%D0%B0_%D0%BF%D1%80%D0%BE%D0%BF%D0%BE%D1%80%D1%86%D1%96%D0%B9_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Тести для малих вибірок та пропорцій



**Завдання 1**. E-commerce компанія після редизайну сайту підозрює, що **середній час до покупки (time-to-purchase)** користувачів **збільшився**.
Історично середній час від першого заходу на сайт до покупки становив $\mu_0 = 123.7$ хвилин.

Команда аналітиків випадково обрала дані **7 покупців після редизайну**:
`128, 135, 121, 142, 126, 151, 123`

З допомогою наявних даних зʼясуйте, чи збільшився середній час покупки після редизайну.

Для цього виконайте наступні 6 кроків. Правильне виконання кожного з кроків оцінюється в 1 бал.

1. Запишіть нульову та альтернативну гіпотези і визначте тип тесту.
2. Обчисліть вибіркові статистики: $\bar x$, $s$, $n$.
3. Оберіть тип тесту та виконайте його (знайдіть критичне значення тесту, статистику тесту та р-значення) будь-яким пасуючим способом, щоб перевірити гіпотезу на рівні значущості $\alpha = 0.10$.
4. Прийміть рішення, чи відхиляєте ви гіпотезу $H_0$ використовуючи p-value.
5. Напишіть висновок: чи справді редизайн сайту подовжив час до покупки?
6. Чи зміниться ваше рішення при зміні рівня значущості на $\alpha = 0.05$.


In [2]:
import numpy as np
from scipy import stats

# Дані з умови
time_to_purchase = np.array([128, 135, 121, 142, 126, 151, 123])
mu0 = 123.7
alpha = 0.10

# Крок 1. Обчислити вибіркову статистику
x_bar = np.mean(time_to_purchase)
s = np.std(time_to_purchase, ddof=1)
n = len(time_to_purchase)

# Крок 2. Обчислити t-статистику вручну
t_stat = (x_bar - mu0) / (s / np.sqrt(n))

# Крок 3. Критичне значення для правостороннього тесту (alternative='greater')
df = n - 1
t_crit = stats.t.ppf(1 - alpha, df)

# Крок 4. p-value (правосторонній тест)
p_value = 1 - stats.t.cdf(t_stat, df)

print(f"Вибіркове середнє: {x_bar:.2f}")
print(f"Вибіркове стандартне відхилення: {s:.2f}")
print(f"t-статистика: {t_stat:.3f}")
print(f"Критичне значення (df={df}): {t_crit:.3f}")
print(f"p-value: {p_value:.4f}")

Вибіркове середнє: 132.29
Вибіркове стандартне відхилення: 10.98
t-статистика: 2.069
Критичне значення (df=6): 1.440
p-value: 0.0420


In [3]:
# Висновок
if t_stat > t_crit:
    print(f"Відхиляємо H0: середній час збільшився на рівні значущості {alpha}")
else:
    print("Не відхиляємо H0: доказів недостатньо")


Відхиляємо H0: середній час збільшився на рівні значущості 0.1


Те саме з використанням лише 1 методу з бібліотеки scipy, який одразу вертає і статистику тесту, і р-значення.

In [4]:
res = stats.ttest_1samp(time_to_purchase, popmean=mu0, alternative="greater")
print(f"\nПеревірка scipy: t = {res.statistic:.3f}, p-value = {res.pvalue:.4f}")


Перевірка scipy: t = 2.069, p-value = 0.0420



# ВИСНОВОК


Оскільки значення p-value = 0.0420 є меншим за рівень значущості $\alpha = 0.10$, відхиляється нульова гіпотеза $H_0$.


Так, редизайн подовжив середній час покупки. Це стверджують дані, на основі яких видно що середній час є на рівні значущості 0,1.
Вибіркове середнє $132.29$ хв виявилося суттєво більшим за історичне $123.7$ хв, і цей результат не може бути випадковістю.


Після зміни рівня значущості на 0.05 рішення не зміниться.
Через те що p-value 0.0420 менше за 0.05, нульова гіпотеза $H_0$ відхиляється.

  



**Завдання 2.**

До спеціальної рекламної кампанії **23%** дорослих упізнавали логотип компанії. Після завершення кампанії відділ маркетингу провів опитування: з **1200** випадково відібраних дорослих **311** упізнали логотип.

Перевірте на рівні значущості **3%** ($\alpha=0.03$), чи дають ці дані достатні підстави стверджувати, що **тепер більше ніж 23%** дорослих упізнають логотип компанії. Для розвʼязку використовуйте бібліотеку `statsmodels`.

Зробіть висновок, чи зросла впізнаваність логотипу.

Додатково, обчисліть довірчий інтревал на заданому рівні значущості і проінтерпретуйте текстом - як він додатково пояснює прийняте нами рішення?

In [5]:
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

count = 311      # кількість тих, хто впізнав
nobs = 1200      # загальна кількість опитаних
p0 = 0.23        # історична пропорція (H0)
alpha_val = 0.03

# z-тест для пропорції (alternative='larger')
stat, pval = proportions_ztest(count, nobs, p0, alternative='larger')
# Обчислення довірчого інтервалу
confint = proportion_confint(count, nobs, alpha=alpha_val, method='normal')

print("🔹 Тест для пропорції впізнаваності")
print(f"Z-статистика = {stat:.3f}")
print(f"p-value = {pval:.4f}")
print(f"{100*(1-alpha_val):.0f}% довірчий інтервал: {confint}")

🔹 Тест для пропорції впізнаваності
Z-статистика = 2.306
p-value = 0.0106
97% довірчий інтервал: (0.23171700302179205, 0.28661633031154127)


In [6]:
# Висновок на основі р-значення
if pval < alpha_val:
    print(f"Відхиляємо H0: впізнаваність логотипу стала більшою за 23% на рівні значущості {alpha_val}")
else:
    print("Не відхиляємо H0: доказів недостатньо")

Відхиляємо H0: впізнаваність логотипу стала більшою за 23% на рівні значущості 0.03


# ІНТЕРПРЕТАЦІЯ

Односторонній тест

Гіпотези: $H_0: p = 0.23$ проти $H_1: p > 0.23$.

p-value = 0.0106 < 0.03, відхиляється нульова гіпотеза $H_0$.


Висновок: статистика показує, що впізнаваність логотипу компанії зросла і тепер більша за 23%. Рекламна кампанія виявилася ефективною.




# ДОВІРЧИЙ ІНТЕРВАЛ

 97% довірчий інтервал (0.23171700302179205, 0.28661633031154127).

Це означає, що з імовірністю 97% частка людей, які впізнають логотип, є у діапазоні 23.19% до 28.64%.

Рішення - відхилити $H_0$: нова впізнаваність вища за попередню.

 Нижня межа інтервалу 23.19% вже перевищує початкові 23%. Це підтверджує моє рішення.